In [ ]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


In [ ]:
!pip install -qU langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 34.6 MB/s eta 0:00:00


In [ ]:
!pip install -qU langfuse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 9.6 MB/s eta 0:00:00


In [ ]:
!pip install -qU langfuse langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.5 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

groq = userdata.get("groq_may")
os.environ["GROQ_API_KEY"] = groq

In [ ]:
Langfuse_S_key = userdata.get("adv_lf")
Langfuse_P_key= userdata.get("adv_lfp")

In [ ]:
os.environ["LANGFUSE_SECRET_KEY"] = Langfuse_S_key
os.environ["LANGFUSE_PUBLIC_KEY"] = Langfuse_P_key
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"

In [ ]:
#LANGFUSE_SECRET_KEY="sk-lf-aac54b0d-520c-4208-8683-6d346d761c84"
#LANGFUSE_PUBLIC_KEY="pk-lf-6ff39959-fa7c-49b4-894c-154fbd26fed3"
#LANGFUSE_BASE_URL="https://us.cloud.langfuse.com"

In [ ]:
from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    reasoning_format="hidden"
)

In [ ]:
llm

ChatGroq(profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7849e21f5910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7849e2216ae0>, model_name='qwen/qwen3-32b', reasoning_format='hidden', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
!pip install -qU langchain-core

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [ ]:

class ExpenseCategoryOutput(BaseModel):
    expense_category: str = Field(
        description="Expense category"
    )

class ExpenseTypeOutput(BaseModel):
    expense_type: str = Field(
        description="Essential, Non-Essential, Uncertain"
    )

class ConfidenceOutput(BaseModel):
    confidence_note: str = Field(
        description="Reasoning note"
    )

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

category_parser = PydanticOutputParser(
    pydantic_object=ExpenseCategoryOutput
)

type_parser = PydanticOutputParser(
    pydantic_object=ExpenseTypeOutput
)

confidence_parser = PydanticOutputParser(
    pydantic_object=ConfidenceOutput
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

category_prompt = ChatPromptTemplate.from_template(
"""
Classify the expense into ONE category.

Categories:
- Food & Groceries
- Transportation
- Utilities
- Housing
- Healthcare
- Entertainment
- Shopping
- Education
- Travel
- Miscellaneous

Expense:
{expense_text}

{format_instructions}
""",
partial_variables={
    "format_instructions":
    category_parser.get_format_instructions()
}
)

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
category_chain = (
    category_prompt
    | llm
    | category_parser
    | RunnableLambda(
        lambda x: x.expense_category
    )
)

In [ ]:
type_prompt = ChatPromptTemplate.from_template(
"""
Determine spending type.

Labels:
- Essential
- Non-Essential
- Uncertain

Rules:
Essential:
Food, rent, utility bills, fuel,
medical expenses, education.

Non-Essential:
Luxury items, entertainment,
premium subscriptions, gadgets.

Uncertain:
Not enough information.

Expense:
{expense_text}

{format_instructions}
""",
partial_variables={
    "format_instructions":
    type_parser.get_format_instructions()
}
)

In [ ]:
type_chain = (
    type_prompt
    | llm
    | type_parser
    | RunnableLambda(
        lambda x: x.expense_type
    )
)

In [ ]:
confidence_prompt = ChatPromptTemplate.from_template(
"""
Generate a short confidence note.

Rules:
1. Explain classification briefly.
2. Maximum one sentence.
3. If description is ambiguous,
   mention uncertainty.

Expense:
{expense_text}

{format_instructions}
""",
partial_variables={
    "format_instructions":
    confidence_parser.get_format_instructions()
}
)

In [ ]:
confidence_chain = (
    confidence_prompt
    | llm
    | confidence_parser
    | RunnableLambda(
        lambda x: x.confidence_note
    )
)

In [ ]:
from langchain_core.runnables import RunnableParallel

expense_analysis = RunnableParallel({"expense_category": category_chain,
                                     "expense_type": type_chain,
                                     "confidence_note": confidence_chain})

In [ ]:
expense_text1= """Paid monthly electricity bill"""
expense_text2 = """Bought groceries from supermarket"""
expense_text3= """Bought premium smartwatch"""

In [ ]:
result = expense_analysis.invoke(
{"expense_text": expense_text1},
config={
        "callbacks": [langfuse_handler]
        })

In [ ]:
result

{'expense_category': 'Utilities',
 'expense_type': 'Essential',
 'confidence_note': 'Classified as an operational expense under utility costs with high confidence due to clear description.'}

In [ ]:

result1 = expense_analysis.invoke(
{ "expense_text": expense_text2},
config={
    "callbacks": [langfuse_handler]
}
)

In [ ]:
result1

{'expense_category': 'Food & Groceries',
 'expense_type': 'Essential',
 'confidence_note': "Classified as 'groceries' with high confidence due to clear purchase context from a supermarket."}

In [ ]:

result3 = expense_analysis.invoke(
{"expense_text": expense_text3},
config={
    "callbacks": [langfuse_handler]
}
)


In [ ]:
result3

{'expense_category': 'Shopping',
 'expense_type': 'Non-Essential',
 'confidence_note': 'Classified as a technology/personal item purchase; uncertainty exists if it serves a business purpose.'}

In [ ]:
#Checks whether confidence note is meaningful